## 准备数据

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [7]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # MNIST: 输入 784 维，输出 10 类；隐藏层 256 维
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1), trainable=True)
        self.b1 = tf.Variable(tf.zeros([256]), trainable=True)
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1), trainable=True)
        self.b2 = tf.Variable(tf.zeros([10]), trainable=True)
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        # x: (batch, 28, 28) -> (batch, 784)
        x = tf.reshape(tf.cast(x, tf.float32), [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)  # (batch, 256)
        logits = tf.matmul(h1, self.W2) + self.b2         # (batch, 10)
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.6030328 ; accuracy 0.1011
epoch 1 : loss 2.3367093 ; accuracy 0.16225
epoch 2 : loss 2.1117857 ; accuracy 0.25066668
epoch 3 : loss 1.9128038 ; accuracy 0.35156667
epoch 4 : loss 1.7336016 ; accuracy 0.462
epoch 5 : loss 1.5717691 ; accuracy 0.5619
epoch 6 : loss 1.4263943 ; accuracy 0.63025
epoch 7 : loss 1.2968478 ; accuracy 0.67725
epoch 8 : loss 1.1822226 ; accuracy 0.71276665
epoch 9 : loss 1.0815306 ; accuracy 0.7388833
epoch 10 : loss 0.9935964 ; accuracy 0.75988334
epoch 11 : loss 0.91703165 ; accuracy 0.7753
epoch 12 : loss 0.85031205 ; accuracy 0.7885
epoch 13 : loss 0.7919599 ; accuracy 0.80065
epoch 14 : loss 0.7405913 ; accuracy 0.81058335
epoch 15 : loss 0.6951401 ; accuracy 0.81981665
epoch 16 : loss 0.65486956 ; accuracy 0.8279167
epoch 17 : loss 0.61922216 ; accuracy 0.83565
epoch 18 : loss 0.587705 ; accuracy 0.84255
epoch 19 : loss 0.55986524 ; accuracy 0.84893334
epoch 20 : loss 0.5352582 ; accuracy 0.85476667
epoch 21 : loss 0.5134778 ; accuracy 0.